# FAO Food Price Index (FFPI) — data exploration
Source file: `data/ffpi-data-2026-11.xlsx` (FAO workbook, downloaded in November 2026 by `data/collect_data.py`).
Goal: understand the workbook layout, check its quality and decide what needs cleaning later. Nothing is dropped or changed in the data here.

In [1]:
# Core libraries for cleaning and exploring data
import pandas as pd
import numpy as np
import pyarrow
# The workbook has several sheets, so open it once and read each sheet from it.
FILE = "../data/ffpi-data-2026-11.xlsx"
xl = pd.ExcelFile(FILE)
print(xl.sheet_names)

['Indices_Monthly', 'Annual', 'Indices_Monthly_Real', 'Annual_Real']


### loading the workbook
- The file is an **Excel workbook with 4 sheets**, so it is opened once with `pd.ExcelFile` and each sheet is read from it.
- `Indices_Monthly` and `Annual` hold the **nominal** index; `Indices_Monthly_Real` and `Annual_Real` hold the **real** index (inflation-adjusted).

In [2]:
# Read every sheet exactly as it is (no header), to see the real layout
raw = {s: xl.parse(s, header=None) for s in xl.sheet_names}
for s, d in raw.items():
    print(s, d.shape)

# The first rows of the main sheet: titles come first, the header is on row 3
raw["Indices_Monthly"].iloc[:6, :8]

Indices_Monthly (445, 66)
Annual (444, 7)
Indices_Monthly_Real (445, 8)
Annual_Real (444, 7)


,0,1,2,3,4,5,6,7
0,FAO Food Price Index,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2014-2016=100,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Date,Food Price Index,Meat,Dairy,Cereals,Oils,Sugar,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1990-01-01 00:00:00,64.444201,74.272219,53.503031,64.140607,44.587672,87.877833,NaN
5,1990-02-01 00:00:00,64.728299,76.781027,52.218634,62.222377,44.500514,90.662693,NaN


### the real layout of a sheet
- Reading without a header shows that the sheets are **report-style**, not tables: row 1 is a title, row 2 gives the base period (`2014-2016=100`), row 3 is the header, row 4 is **empty**, and the data starts on row 5.
- `Indices_Monthly` has **66 columns** but only 7 are real (date + the headline index + 5 sub-indices: Meat, Dairy, Cereals, Oils, Sugar).

In [3]:
# Which columns of the main sheet actually hold data?
d = raw["Indices_Monthly"]
filled = d.notna().sum()
print("Columns with any value:", filled[filled > 0].to_dict())

# Two unlabeled columns (36 and 65) are almost empty / unrelated to the 7 real columns
print("Column 36:", d[36].dropna().to_dict())
print("Column 65: %d values, from row %d to row %d, range %.1f to %.1f" % (
    d[65].notna().sum(), d[65].first_valid_index(), d[65].last_valid_index(), d[65].min(), d[65].max()))

# Rows where the first column is empty, and the last rows of every sheet
for s, x in raw.items():
    last_rows = x.dropna(how="all").tail(2)
    print(s, "-> last non-empty rows:")
    print(last_rows.iloc[:, :3].to_string(header=False))

Columns with any value: {0: 444, 1: 442, 2: 442, 3: 442, 4: 442, 5: 442, 6: 442, 36: 1, 65: 297}
Column 36: {386: 18.2225}
Column 65: 297 values, from row 122 to row 418, range 61.6 to 132.8
Indices_Monthly -> last non-empty rows:
443  2026-08-01 00:00:00  133.980459  129.309472
444  2026-09-01 00:00:00  135.962883  127.856984
Annual -> last non-empty rows:
39                  2026  130.221181  128.686994
443  2026-08-01 00:00:00         NaN         NaN
Indices_Monthly_Real -> last non-empty rows:
443  2026.0  2026-08-01 00:00:00  115.457601
444  2026.0  2026-09-01 00:00:00  117.165954
Annual_Real -> last non-empty rows:
439  2026-04-01 00:00:00  NaN  NaN
443  2026-08-01 00:00:00  NaN  NaN


### stray cells outside the table
- Columns 0 to 6 hold the data. **Column 36 has one stray number** (`18.22`, at row index 386) and **column 65 has 297 unlabeled values** (between `61.6` and `132.8`, row indexes 122 to 418). Neither belongs to the table, so they are left out when reading.
- The two **Annual sheets end with stray date cells** far below the data (for example `2026-08-01` at row index 443, with no values next to it).
- The other sheets end cleanly on their last real month (2026-09).

In [4]:
# Read the main sheet with the header on row 3 and only the 7 real columns (A:G)
df_raw = pd.read_excel(FILE, sheet_name="Indices_Monthly", header=2, usecols="A:G")
print("Shape as read:", df_raw.shape)

# Rows that are completely empty except for the date (stray cell below the data)
print(df_raw[df_raw["Food Price Index"].isna()])

Shape as read: (442, 7)
  Date  Food Price Index  Meat  Dairy  Cereals  Oils  Sugar
0  NaT               NaN   NaN    NaN      NaN   NaN    NaN


### reading the main sheet properly
- `header=2` puts the header on row 3 and `usecols="A:G"` keeps only the 7 real columns. Result: **(442, 7)**.
- One of the 442 rows is empty except for the date column (`NaT`): it is the **blank row under the header**.

In [5]:
#copy the data frame to a new variable to avoid changing the original data frame
# and drop the stray row that has a date but no values
df = df_raw.dropna(subset=["Food Price Index"]).copy()
print("Shape:", df.shape)
df.head()

Shape: (441, 7)


,Date,Food Price Index,Meat,Dairy,Cereals,Oils,Sugar
1,1990-01-01,64.444201,74.272219,53.503031,64.140607,44.587672,87.877833
2,1990-02-01,64.728299,76.781027,52.218634,62.222377,44.500514,90.662693
3,1990-03-01,64.033021,78.543983,41.367123,61.259884,45.745434,95.056585
4,1990-04-01,66.015997,81.190035,48.427068,62.820731,44.017094,94.313956
5,1990-05-01,64.632844,81.756679,39.189651,61.982826,45.499631,90.353265


### working copy
- Dropping the blank row leaves **441 rows x 7 columns**.
- The series starts in **1990-01**: Food Price Index `64.44`, Meat `74.27`, Dairy `53.50`, Cereals `64.14`, Oils `44.59`, Sugar `87.88` (all with 2014-2016 = 100).

In [6]:
df.tail(3)

,Date,Food Price Index,Meat,Dairy,Cereals,Oils,Sugar
439,2026-07-01,131.708669,129.410004,116.552147,113.767069,195.727808,95.045822
440,2026-08-01,133.980459,129.309472,119.223976,116.766441,196.858022,107.410169
441,2026-09-01,135.962883,127.856984,119.050790,122.762461,198.628322,113.954553


### end of the series
- The last month is **2026-09**: Food Price Index `135.96`, Meat `127.86`, Dairy `119.05`, Cereals `122.76`, Oils `198.63`, Sugar `113.95`.
- The README says the file ends in 2026-08, so this workbook has **one more month** than the README states.

In [7]:
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 441 entries, 1 to 441
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Date              441 non-null    datetime64[us]
 1   Food Price Index  441 non-null    float64       
 2   Meat              441 non-null    float64       
 3   Dairy             441 non-null    float64       
 4   Cereals           441 non-null    float64       
 5   Oils              441 non-null    float64       
 6   Sugar             441 non-null    float64       
dtypes: datetime64[us](1), float64(6)
memory usage: 24.2 KB


### data types and memory
- `Date` is already `datetime64` and the six indices are `float64`: **pandas converted the types correctly**, so no type fix is needed.
- All 7 columns have **441 non-null** values and the DataFrame uses only about **24 KB**.

In [8]:
# Null counts, duplicates and month continuity
print("Nulls per column:")
print(df.isna().sum())
print("Duplicate rows :", df.duplicated().sum())
print("Duplicate dates:", df["Date"].duplicated().sum())

expected = pd.date_range(df["Date"].min(), df["Date"].max(), freq="MS")
print("Range   :", df["Date"].min().date(), "->", df["Date"].max().date())
print("Expected:", len(expected), " Found:", len(df), " Missing month-start dates:", len(expected.difference(df["Date"])))

# Those 'missing' months are not missing: the date is simply not on day 1
print("Rows not dated on day 1:", (df["Date"].dt.day != 1).sum())
print(df.loc[df["Date"].dt.day != 1, "Date"].dt.strftime("%Y-%m-%d").tolist())

# Check by calendar month instead of by exact date
print("Distinct calendar months:", df["Date"].dt.to_period("M").nunique(), " Duplicated months:", df["Date"].dt.to_period("M").duplicated().sum())

Nulls per column:
Date                0
Food Price Index    0
Meat                0
Dairy               0
Cereals             0
Oils                0
Sugar               0
dtype: int64
Duplicate rows : 0
Duplicate dates: 0
Range   : 1990-01-01 -> 2026-09-01
Expected: 441  Found: 441  Missing month-start dates: 7
Rows not dated on day 1: 7
['2017-06-02', '2018-03-02', '2018-04-02', '2018-05-03', '2018-07-02', '2019-06-02', '2019-11-02']
Distinct calendar months: 441  Duplicated months: 0


### missing values, duplicates and month continuity
- **No nulls, no duplicate rows and no duplicate dates.** The file covers **441 months, from 1990-01 to 2026-09**.
- A strict check for month-start dates reports **7 "missing" months**, but nothing is missing: **7 rows are dated on the 2nd or 3rd** instead of the 1st (2017-06-02, 2018-03-02, 2018-04-02, 2018-05-03, 2018-07-02, 2019-06-02 and 2019-11-02).
- Counted by calendar month, there are **441 distinct months and 0 duplicates**.
- **Implication:** the date must be moved to the first day of its month before joining with other files, otherwise these 7 months would not match.

In [9]:
# Summary stats for the numeric columns (2014-2016 = 100)
df.describe().round(1)

,Date,Food Price Index,Meat,Dairy,Cereals,Oils,Sugar
count,441,441.0,441.0,441.0,441.0,441.0,441.0
mean,2008-05-01 10:36:44.081632,90.0,88.3,89.3,90.1,95.6,85.1
min,1990-01-01 00:00:00,50.8,52.3,36.8,48.6,35.8,31.8
25%,1999-03-01 00:00:00,64.7,73.6,57.7,60.7,64.2,60.4
50%,2008-05-01 00:00:00,89.1,85.6,83.5,89.8,84.3,79.3
75%,2017-07-01 00:00:00,118.6,103.5,118.4,112.1,121.5,107.9
max,2026-09-01 00:00:00,160.2,131.3,158.2,173.5,251.8,183.2
std,NaN,27.6,19.2,34.9,31.5,41.8,32.4


### numeric summary
- The headline **Food Price Index** has a mean of `90.0`, a minimum of `50.8` and a maximum of `160.2` (std `27.6`).
- **Oils is the most volatile sub-index** (`35.8` to `251.8`, std `41.8`), followed by Dairy (std `34.9`), Sugar (`32.4`) and Cereals (`31.5`). **Meat is the steadiest** (std `19.2`).

In [10]:
# Is the base period really 100? Average of the three base years, per index
base = df[df["Date"].dt.year.between(2014, 2016)]
base[["Food Price Index", "Meat", "Dairy", "Cereals", "Oils", "Sugar"]].mean().round(2)

Food Price Index    100.0
Meat                100.0
Dairy               100.0
Cereals             100.0
Oils                100.0
Sugar               100.0
dtype: float64

### is the base period really 100?
- The 2014 to 2016 average is **`100.0` for all six indices**, which confirms the base period. A value above 100 means prices above the 2014-2016 level.

In [11]:
# Is the headline index a weighted average of the five sub-indices?
subs = ["Meat", "Dairy", "Cereals", "Oils", "Sugar"]
w, *_ = np.linalg.lstsq(df[subs].values, df["Food Price Index"].values, rcond=None)
print("Implied weights:", dict(zip(subs, w.round(3))), " Sum:", w.sum().round(3))
print("Max |FFPI - weighted sum|:", np.abs(df[subs].values @ w - df["Food Price Index"].values).max().round(4))

Implied weights: {'Meat': np.float64(0.333), 'Dairy': np.float64(0.14), 'Cereals': np.float64(0.289), 'Oils': np.float64(0.167), 'Sugar': np.float64(0.071)}  Sum: 1.0
Max |FFPI - weighted sum|: 0.0


### how the headline index is built
- The Food Price Index is an **exact weighted average of the five sub-indices**: Meat `0.333`, Dairy `0.140`, Cereals `0.289`, Oils `0.167`, Sugar `0.071` (sum `1.0`, largest error `0.0`).
- These weights are recovered by regression, not read from FAO, but they reproduce the index perfectly.
- **Implication:** the headline index adds no new information beyond the five sub-indices.

In [12]:
# Highest and lowest month of every index
rows = []
for c in ["Food Price Index"] + subs:
    rows.append([c, df.loc[df[c].idxmax(), "Date"].strftime("%Y-%m"), round(df[c].max(), 1),
                 df.loc[df[c].idxmin(), "Date"].strftime("%Y-%m"), round(df[c].min(), 1),
                 round(df[c].iloc[-1], 1)])
pd.DataFrame(rows, columns=["index", "peak month", "peak", "lowest month", "lowest", "latest (2026-09)"])

,index,peak month,peak,lowest month,lowest,latest (2026-09)
0,Food Price Index,2022-03,160.2,2002-05,50.8,136.0
1,Meat,2026-05,131.3,2002-12,52.3,127.9
2,Dairy,2022-06,158.2,1990-08,36.8,119.1
3,Cereals,2022-05,173.5,2000-08,48.6,122.8
4,Oils,2022-03,251.8,2001-02,35.8,198.6
5,Sugar,2011-01,183.2,2000-03,31.8,114.0


### peaks and lows
| Index | Peak month | Peak | Lowest month | Lowest | Latest (2026-09) |
|---|---|---|---|---|---|
| Food Price Index | 2022-03 | 160.2 | 2002-05 | 50.8 | 136.0 |
| Meat | 2026-05 | 131.3 | 2002-12 | 52.3 | 127.9 |
| Dairy | 2022-06 | 158.2 | 1990-08 | 36.8 | 119.1 |
| Cereals | 2022-05 | 173.5 | 2000-08 | 48.6 | 122.8 |
| Oils | 2022-03 | 251.8 | 2001-02 | 35.8 | 198.6 |
| Sugar | 2011-01 | 183.2 | 2000-03 | 31.8 | 114.0 |

- Four of the six indices peaked in **2022** (the post-pandemic and Ukraine war spike).
- **Meat reached its all-time high in 2026-05**, in the current year.
- **Oils is still about 99% above its 2014-2016 level** (`198.6`) even though it is below its 2022 peak, and Cereals has risen in the last months (`110.0` in 2026-06 to `122.8` in 2026-09).

In [13]:
# The Annual sheet: header on row 3, 7 columns, drop the empty rows below the data
annual_raw = pd.read_excel(FILE, sheet_name="Annual", header=2, usecols="A:G")
annual = annual_raw.dropna(subset=["Food Price Index"]).copy()
print("Read:", annual_raw.shape, " After dropping empty rows:", annual.shape)
print(annual_raw["Year"].map(type).value_counts())

# Does the annual value equal the mean of the 12 monthly values?
m = df.assign(Year=df["Date"].dt.year).groupby("Year")
chk = m["Food Price Index"].mean().to_frame("monthly mean").join(annual.set_index("Year")["Food Price Index"].rename("annual sheet"))
chk["months"] = m.size()
print("Max difference:", (chk["monthly mean"] - chk["annual sheet"]).abs().max().round(4))
chk.tail(3)

Read: (441, 7)  After dropping empty rows: (37, 7)
Year
<class 'float'>                403
<class 'int'>                   37
<class 'datetime.datetime'>      1
Name: count, dtype: int64
Max difference: 0.0


,monthly mean,annual sheet,months
Year,,,
2024,121.981996,121.981996,12
2025,127.191617,127.191617,12
2026,130.221181,130.221181,9


### the Annual sheet
- Reading it the same way gives 441 rows, but only **37 are real years (1990 to 2026)**: the rest are empty rows and the stray date cells. The `Year` column mixes integers, empty cells and one date, which is why it must be cleaned.
- Each annual value is **exactly the mean of the monthly values** (largest difference `0.0`).
- **2026 is a partial year**: its `130.2` is the average of 9 months (January to September), so it should not be compared with full years.

In [14]:
# The Real sheets: same layout, but the date sits in column B and column A is the year
real_raw = pd.read_excel(FILE, sheet_name="Indices_Monthly_Real", header=2, usecols="B:H")
real = real_raw.dropna(subset=["Food Price Index"]).copy()
real.columns = ["Date", "Food Price Index", "Meat", "Dairy", "Cereals", "Oils", "Sugar"]
print("Read:", real_raw.shape, " After dropping empty rows:", real.shape)
print("Range:", real["Date"].min().date(), "->", real["Date"].max().date())
print("Rows not dated on day 1:", (real["Date"].dt.day != 1).sum(), "(same dates as the nominal sheet:", bool((real["Date"].values == df["Date"].values).all()), ")")

# Real / nominal = how much the index was deflated, month by month
ratio = (real.set_index("Date")["Food Price Index"] / df.set_index("Date")["Food Price Index"])
print("Real / nominal in 1990-01:", ratio.iloc[0].round(3), " in 2014-2016 (mean):", ratio["2014":"2016"].mean().round(3), " in 2026-09:", ratio.iloc[-1].round(3))
print("Nominal ends 2026-09 at %.1f, real ends at %.1f" % (df["Food Price Index"].iloc[-1], real["Food Price Index"].iloc[-1]))

Read: (442, 7)  After dropping empty rows: (441, 7)
Range: 1990-01-01 -> 2026-09-01
Rows not dated on day 1: 7 (same dates as the nominal sheet: True )
Real / nominal in 1990-01: 1.21  in 2014-2016 (mean): 1.004  in 2026-09: 0.862
Nominal ends 2026-09 at 136.0, real ends at 117.2


### the Real sheet
- It has the same 441 months (1990-01 to 2026-09) and the **same 7 off-day dates**, but the date is in column B and column A repeats the year, so it is read with `usecols="B:H"`.
- **Real / nominal** is `1.21` in 1990-01, about `1.00` in 2014-2016 (the base period) and `0.86` in 2026-09. In 2026-09 the nominal index is `136.0` and the real index is `117.2`.
- So for comparisons over time the real series is the fair one: part of the nominal rise is just inflation.

In [15]:
# Compare the peaks: nominal vs real (is 2022 still the worst year once inflation is removed?)
for name, x in [("Nominal", df), ("Real", real)]:
    top = x.nlargest(3, "Food Price Index")[["Date", "Food Price Index"]]
    print(name, "top 3 months:", [(d.strftime("%Y-%m"), round(v, 1)) for d, v in zip(top["Date"], top["Food Price Index"])])
print("Real index in 1990-1992 (mean):", real[real["Date"].dt.year.between(1990, 1992)]["Food Price Index"].mean().round(1))
print("Real index in 2026-09:", real["Food Price Index"].iloc[-1].round(1))

Nominal top 3 months: [('2022-03', 160.2), ('2022-04', 158.8), ('2022-05', 158.7)]
Real top 3 months: [('2022-03', 136.5), ('2022-04', 135.3), ('2022-05', 135.2)]
Real index in 1990-1992 (mean): 76.6
Real index in 2026-09: 117.2


### nominal against real
- **2022-03 is the highest month in both series** (nominal `160.2`, real `136.5`), followed by 2022-04 and 2022-05.
- In real terms the index averaged `76.6` in 1990 to 1992 and is `117.2` now, about **53% higher**. Part of the nominal rise is only inflation, which is why the real series is the one to use for comparisons.

In [16]:
# Save Parquet copies of the four sheets (read with the header fix only; no cleaning of values)
import os
os.makedirs("../data/parquet", exist_ok=True)
out = {
    "ffpi_monthly_raw":      df_raw,
    "ffpi_annual_raw":       annual_raw,
    "ffpi_monthly_real_raw": real_raw,
    "ffpi_annual_real_raw":  pd.read_excel(FILE, sheet_name="Annual_Real", header=2, usecols="A:G"),
}
for name, frame in out.items():
    frame = frame.copy()
    first = frame.columns[0]
    if frame[first].dtype == "object":                 # the Annual sheets mix years, dates and empty cells
        frame[first] = frame[first].map(lambda v: None if pd.isna(v) else str(v))
    frame.to_parquet(f"../data/parquet/{name}.parquet")
    print(f"{name:24s}", frame.shape, round(os.path.getsize(f'../data/parquet/{name}.parquet') / 1024, 1), "KB")
print("Excel:", round(os.path.getsize(FILE) / 1024, 1), "KB")

ffpi_monthly_raw         (442, 7) 32.3 KB
ffpi_annual_raw          (441, 7) 7.4 KB
ffpi_monthly_real_raw    (442, 7) 32.9 KB


ffpi_annual_real_raw     (441, 7) 7.5 KB
Excel: 123.5 KB


### Parquet checkpoint
- The four sheets are saved in `data/parquet/` (`ffpi_monthly_raw`, `ffpi_annual_raw`, `ffpi_monthly_real_raw`, `ffpi_annual_real_raw`).
- They only have the header fix and the 7 real columns; **no values are cleaned**. The mixed first column of the Annual sheets is stored as text.
- The four files total about **80 KB** against **123.5 KB** for the Excel workbook; cleaning is done later in SQL.

### Conclusion: coverage and data quality

| Check | Result |
|---|---|
| Sheets | 4 (monthly and annual, nominal and real) |
| Monthly period | 1990-01 to 2026-09 (441 months) |
| Annual period | 1990 to 2026 (2026 is a partial year, 9 months) |
| Base period | 2014-2016 = 100 (confirmed for all indices) |
| Missing values / duplicates | none / none |
| Dates not on day 1 | 7 (2017-06, 2018-03, 2018-04, 2018-05, 2018-07, 2019-06, 2019-11) |

### Conclusion: decisions for the cleaning step

| Topic | Action |
|---|---|
| Header and columns | Header on row 3, data from row 5; keep only the 7 real columns (drop the unlabeled columns 36 and 65) |
| Empty and stray rows | Drop the blank row under the header and the stray date cells in the Annual sheets |
| Dates | Move every date to the first day of its month before any join |
| Headline index | Keep it for display, but remember it is a weighted sum of the five sub-indices (do not treat it as independent) |
| Nominal or real | Use `Indices_Monthly_Real` for comparisons over time (as the README says) |
| Annual 2026 | Partial year: exclude it or flag it |
| README | Update the end date: the file now ends in **2026-09**, not 2026-08 |